In [1]:
import torch as tch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets
from torchvision.transforms import v2


In [2]:
train_dataset = datasets.EMNIST(root="data", split="bymerge", train=True, download=True, transform=v2.Compose([v2.ToImage(),v2.ToDtype(dtype=tch.float32, scale=True)]))
size = len(train_dataset)
print(size)
train_size = int(0.8 * size)
val_size = size - train_size
train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size]) # Splits the training dataset into an 80/20 split of training/validation
test_dataset = datasets.EMNIST(root="data", split="bymerge", train=False, download=True, transform=v2.Compose([v2.ToImage(),v2.ToDtype(dtype=tch.float32, scale=True)]))

100.0%


697932


In [3]:
train_dataloader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size =256, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size = 256, shuffle=True)

In [4]:
class Model(tch.nn.Module):
    
    def __init__(self):
        super().__init__()
        self.prelu1 = tch.nn.PReLU()
        self.prelu2 = tch.nn.PReLU()
        self.prelu3 = tch.nn.PReLU()
        self.prelu4 = tch.nn.PReLU()
        self.conv1 = tch.nn.Conv2d(1, 32, 5)
        self.batchnorm1 = tch.nn.BatchNorm2d(32)
        self.conv2 = tch.nn.Conv2d(32, 64, 5)
        self.batchnorm2 = tch.nn.BatchNorm2d(64)
        self.conv3 = tch.nn.Conv2d(64, 128, 4)
        self.batchnorm3 = tch.nn.BatchNorm2d(128)
        self.avgpool = tch.nn.AdaptiveAvgPool2d((1,1))
        self.fc1 = tch.nn.Linear(128, 64)
        self.fc2 = tch.nn.Linear(64, 47)
    def forward(self, input):
        x = self.conv1(input)
        x = self.batchnorm1(x)
        x = self.prelu1(x)
        x = tch.nn.functional.max_pool2d(x, (2, 2))
        x = self.conv2(x)
        x = self.batchnorm2(x)
        x = self.prelu2(x)
        x = tch.nn.functional.max_pool2d(x, (2, 2))
        x = self.conv3(x)
        x = self.batchnorm3(x)
        x = self.prelu3(x)
        x = self.avgpool(x)
        x = tch.flatten(x, 1)
        x = self.fc1(x)
        x = self.prelu4(x)
        x = self.fc2(x)
        return x

In [5]:
# model_state_dict = tch.load("model.pth", weights_only=False)
model = Model().to("cpu")
# model.load_state_dict(model_state_dict)
loss_fn = tch.nn.CrossEntropyLoss()
optimizer = tch.optim.SGD(lr=0.01, params=model.parameters(), momentum=0.9)
scheduler = tch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer, mode="min", patience=5)
sum(p.numel() for p in model.parameters())


195059

In [6]:
def validation():
    model.eval()
    correct= 0
    val_loss = 0
    size = len(val_dataloader.dataset)
    with tch.no_grad():
        for batch, (x, y) in enumerate(val_dataloader):
            pred= model(x)
            val_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(tch.float).sum().item()
    val_loss /= len(val_dataloader)
    correct /= size
    return val_loss

In [ ]:

model.train()
for i in range(20):
    for batch, (x, y) in enumerate(train_dataloader):
        pred = model.forward(x)
        loss = loss_fn(pred, y)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        print(batch, loss)
    validation_loss = validation()
    scheduler.step(validation_loss)



0 tensor(3.8543, grad_fn=<NllLossBackward0>)
1 tensor(3.8457, grad_fn=<NllLossBackward0>)
2 tensor(3.8144, grad_fn=<NllLossBackward0>)
3 tensor(3.8020, grad_fn=<NllLossBackward0>)
4 tensor(3.7358, grad_fn=<NllLossBackward0>)
5 tensor(3.7083, grad_fn=<NllLossBackward0>)
6 tensor(3.6464, grad_fn=<NllLossBackward0>)
7 tensor(3.6422, grad_fn=<NllLossBackward0>)
8 tensor(3.5807, grad_fn=<NllLossBackward0>)
9 tensor(3.5600, grad_fn=<NllLossBackward0>)
10 tensor(3.5085, grad_fn=<NllLossBackward0>)
11 tensor(3.4130, grad_fn=<NllLossBackward0>)
12 tensor(3.4547, grad_fn=<NllLossBackward0>)
13 tensor(3.3490, grad_fn=<NllLossBackward0>)
14 tensor(3.3286, grad_fn=<NllLossBackward0>)
15 tensor(3.2888, grad_fn=<NllLossBackward0>)
16 tensor(3.2256, grad_fn=<NllLossBackward0>)
17 tensor(3.1597, grad_fn=<NllLossBackward0>)
18 tensor(3.1451, grad_fn=<NllLossBackward0>)
19 tensor(3.1053, grad_fn=<NllLossBackward0>)
20 tensor(2.9740, grad_fn=<NllLossBackward0>)
21 tensor(2.9388, grad_fn=<NllLossBackward0>

KeyboardInterrupt: 

In [8]:
model.eval()
correct= 0
test_loss = 0
size = len(test_dataloader.dataset)
with tch.no_grad():
    for batch, (x, y) in enumerate(test_dataloader):
        pred= model(x)
        test_loss += loss_fn(pred, y).item()
        correct += (pred.argmax(1) == y).type(tch.float).sum().item()

test_loss /= len(test_dataloader)
correct /= size
print("Accuracy:", correct * 100, "Loss:", test_loss)

Accuracy: 90.5521693904043 Loss: 0.27659484301949594


In [10]:
tch.save(model.state_dict(), "./models/model.pth")

In [11]:
sum(p.numel() for p in model.parameters())

195059